# Chapter 29: Multi Session SLAM

<a href="../lite/lab/index.html?path=ch29_multisession_slam.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

Monday: your robot maps the office. Tuesday: it maps it again. The Monday map has errors
in the kitchen. The Tuesday map has errors in the conference room. Can you combine them
so each map fixes the other's weak spots? That is multi session SLAM.

This chapter shows how to merge SLAM sessions that share overlapping areas.

## 29.1 Merging Trajectories

Two sessions share some landmarks. By identifying common features, we can compute the
transform between session coordinate frames and merge them into a single consistent map.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_common = 4           # shared landmarks
n_session1_only = 3    # landmarks only in session 1
n_session2_only = 3    # landmarks only in session 2
# ──────────────────────────────────────────────────────────────────────────────

# Common landmarks
common_lm = np.random.uniform(3, 7, (n_common, 2))
s1_lm = np.vstack([common_lm, np.random.uniform(0, 4, (n_session1_only, 2))])
s2_lm = np.vstack([common_lm + np.random.normal(0, 0.2, (n_common, 2)),
                    np.random.uniform(6, 10, (n_session2_only, 2))])

# Session trajectories
t1 = np.linspace(0, np.pi, 30)
traj1 = np.column_stack([2 + 3*np.cos(t1), 5 + 3*np.sin(t1)])
t2 = np.linspace(np.pi, 2*np.pi, 30)
traj2 = np.column_stack([8 + 3*np.cos(t2), 5 + 3*np.sin(t2)])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.plot(traj1[:, 0], traj1[:, 1], 'steelblue', lw=2, label='Session 1')
ax.scatter(s1_lm[:, 0], s1_lm[:, 1], c='steelblue', s=80, marker='^', zorder=5)
ax.set_title("Session 1", fontsize=13); ax.set_aspect('equal'); ax.legend()
ax.set_xlim(-2, 12); ax.set_ylim(0, 10)

ax = axes[1]
ax.plot(traj2[:, 0], traj2[:, 1], 'tomato', lw=2, label='Session 2')
ax.scatter(s2_lm[:, 0], s2_lm[:, 1], c='tomato', s=80, marker='^', zorder=5)
ax.set_title("Session 2", fontsize=13); ax.set_aspect('equal'); ax.legend()
ax.set_xlim(-2, 12); ax.set_ylim(0, 10)

ax = axes[2]
ax.plot(traj1[:, 0], traj1[:, 1], 'steelblue', lw=2, label='Session 1')
ax.plot(traj2[:, 0], traj2[:, 1], 'tomato', lw=2, label='Session 2')
ax.scatter(common_lm[:, 0], common_lm[:, 1], c='forestgreen', s=120, marker='*', zorder=5, label='shared landmarks')
ax.set_title("Merged", fontsize=13); ax.set_aspect('equal'); ax.legend()
ax.set_xlim(-2, 12); ax.set_ylim(0, 10)

plt.tight_layout()
plt.show()

## 29.2 Cross Session Loops

When session 2 observes the same landmarks as session 1, these create **cross session
loop closures**. They constrain the relative alignment between sessions and improve both maps.

**Key observations:**
- Multi session SLAM requires **place recognition** across sessions.
- Shared landmarks act as anchors connecting the two coordinate frames.
- The merged map is typically better than either individual session.

---

## Exercises

### Exercise 29.1
Simulate two overlapping sessions. Compute the rigid transform (rotation + translation)
between them using shared landmark positions (SVD-based alignment). Apply the transform
and plot the merged result.

In [ ]:
# Your code here